# Model Training: DistilBERT on Banking77

Fine-tuning DistilBERT for banking intent classification.

In [36]:
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import Dataset as HFDataset

# Set random seed for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

## 1. Load Data

In [23]:
# If on Google Colab with Drive mounted:
# DATA_PATH = '/content/drive/MyDrive/Call-Analysis-Platform/AI_Module/data/raw'
# If running locally in notebooks/ folder:

DATA_PATH = '../../AI_Module/data/raw'

df_train = pd.read_csv(f'{DATA_PATH}/banking77-train.csv')
df_test = pd.read_csv(f'{DATA_PATH}/banking77-test.csv')

print(f"Train: {df_train.shape}")
print(f"Test: {df_test.shape}")
print(f"Number of intents: {df_train['label_text'].nunique()}")

Train: (9993, 3)
Test: (3076, 3)
Number of intents: 77


## 2. Create Label Info

In [24]:
# Get number of unique labels and label names (from existing data)
NUM_LABELS = df_train['label'].nunique()
unique_labels = sorted(df_train['label_text'].unique())

print(f"Number of labels: {NUM_LABELS}")
print(f"Sample labels: {unique_labels[:5]}")

Number of labels: 77
Sample labels: ['Refund_not_showing_up', 'activate_my_card', 'age_limit', 'apple_pay_or_google_pay', 'atm_support']


## 3. Tokenization

In [8]:
# Load DistilBERT tokenizer
MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH = 128  # From EDA: max is 79 words, use 128 for safety

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## 4. Prepare Dataset

In [26]:
# Use existing 'label' column (numeric) - rename to 'labels' for HF Trainer
# Stratified train/validation split (80/20)
train_df, val_df = train_test_split(
    df_train[['text', 'label', 'label_text']],
    test_size=0.2,
    stratify=df_train['label'],
    random_state=SEED
)

# Rename 'label' to 'labels' for HuggingFace Trainer
train_df = train_df.rename(columns={'label': 'labels'})
val_df = val_df.rename(columns={'label': 'labels'})

print(f"Train split: {train_df.shape}")
print(f"Validation split: {val_df.shape}")

Train split: (7994, 3)
Validation split: (1999, 3)


In [27]:
# Tokenize function
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=MAX_LENGTH
    )

In [28]:
# Convert to HuggingFace Dataset format (use existing 'label' column, rename to 'labels')
test_df = df_test[['text', 'label']].rename(columns={'label': 'labels'})

train_dataset = HFDataset.from_pandas(train_df[['text', 'labels']])
val_dataset = HFDataset.from_pandas(val_df[['text', 'labels']])
test_dataset = HFDataset.from_pandas(test_df)

# Tokenize
train_tokenized = train_dataset.map(tokenize_function, batched=True)
val_tokenized = val_dataset.map(tokenize_function, batched=True)
test_tokenized = test_dataset.map(tokenize_function, batched=True)

# Set format for PyTorch
train_tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
val_tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
test_tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

print(f"Tokenized train: {len(train_tokenized)}")
print(f"Tokenized val: {len(val_tokenized)}")
print(f"Tokenized test: {len(test_tokenized)}")

Map:   0%|          | 0/7994 [00:00<?, ? examples/s]

Map:   0%|          | 0/1999 [00:00<?, ? examples/s]

Map:   0%|          | 0/3076 [00:00<?, ? examples/s]

Tokenized train: 7994
Tokenized val: 1999
Tokenized test: 3076


## 5. Load Model

In [29]:
# Load DistilBERT for sequence classification
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
)

# Create id2label for inference
id2label = {row['label']: row['label_text'] for _, row in df_train[['label', 'label_text']].drop_duplicates().iterrows()}

print(f"Model parameters: {model.num_parameters():,}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model parameters: 67,012,685


## 6. Training

In [30]:
# Define metrics using HF Evaluate
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    # Macro F1 - critical due to class imbalance
    results = f1_metric.compute(predictions=predictions, references=labels, average='macro')
    return {'macro_f1': results['f1']}

In [40]:
# Training arguments
# UPDATE OUTPUT_DIR for your environment
OUTPUT_DIR = '../../AI_Module/models/distilbert-banking77'

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    save_total_limit=2,
    logging_dir='./logs',
    logging_steps=50,
    seed=SEED,
    fp16=True,  # Enable mixed precision (use GPU)
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [41]:
# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    compute_metrics=compute_metrics,
)

In [42]:
# Train the model
trainer.train()

Epoch,Training Loss,Validation Loss,Macro F1
1,0.507717,0.562899,0.861324
2,0.289157,0.394292,0.894374
3,0.199862,0.376494,0.898271
4,0.099994,0.352786,0.905196
5,0.059752,0.370214,0.905736
6,0.040057,0.398657,0.905167
7,0.023994,0.390249,0.907933
8,0.009078,0.398656,0.907770
9,0.012264,0.404099,0.907270
10,0.011926,0.403214,0.908949


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=5000, training_loss=0.16228733648657798, metrics={'train_runtime': 406.6732, 'train_samples_per_second': 196.571, 'train_steps_per_second': 12.295, 'total_flos': 2650901856476160.0, 'train_loss': 0.16228733648657798, 'epoch': 10.0})

## 7. Evaluation

In [43]:
# Evaluate on validation set
val_results = trainer.evaluate()
print(f"Validation Results:")
print(f"  Macro F1: {val_results['eval_macro_f1']:.4f}")

Validation Results:
  Macro F1: 0.9089


In [44]:
# Evaluate on test set
test_results = trainer.evaluate(eval_dataset=test_tokenized)
print(f"Test Results:")
print(f"  Macro F1: {test_results['eval_macro_f1']:.4f}")

Test Results:
  Macro F1: 0.9245


In [45]:
# Get predictions for detailed analysis
predictions = trainer.predict(test_tokenized)
pred_labels = np.argmax(predictions.predictions, axis=-1)
true_labels = predictions.label_ids

# Classification report
print("Classification Report:")
print(classification_report(
    true_labels,
    pred_labels,
    target_names=unique_labels,
    digits=4
))

Classification Report:
                                                  precision    recall  f1-score   support

                           Refund_not_showing_up     1.0000    0.9750    0.9873        40
                                activate_my_card     1.0000    1.0000    1.0000        40
                                       age_limit     1.0000    1.0000    1.0000        40
                         apple_pay_or_google_pay     0.9750    1.0000    0.9873        39
                                     atm_support     0.9744    0.9500    0.9620        40
                                automatic_top_up     0.7381    0.7750    0.7561        40
         balance_not_updated_after_bank_transfer     0.9750    0.9750    0.9750        40
balance_not_updated_after_cheque_or_cash_deposit     1.0000    0.8750    0.9333        40
                         beneficiary_not_allowed     1.0000    0.9500    0.9744        40
                                 cancel_transfer     0.9756    1.0000    0.9

## 8. Export Model

In [38]:
# Save model and tokenizer
model_path = '../../AI_Module/models/distilbert-banking77'

model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)

print(f"Model saved to: {model_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: /content/
